#### setup

In [ ]:
# make library code importable
import sys
from pathlib import Path
ROOT = Path.cwd().parents[3]
sys.path.append(str(ROOT / "library"))

# generic imports
import os
import json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# library imports
from data_utils import *
from models import *
from training import *
from eval_utils import *

In [ ]:
# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# set confounders
confounders = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11','x12', 'x13', 'x14', 'x15', 
               'x16', 'x17', 'x18', 'x19', 'x20', 'x21','x22', 'x23', 'x24', 'x25']
input_dim = len(confounders)

#### helpers

In [ ]:
def get_nuisance_params(configs, nuisance):
    row = configs.loc[configs["nuisance"] == nuisance].iloc[0]

    return dict(
        hidden_dim=int(row["hidden_dim"]),
        learning_rate=float(row["lr"]),
        weight_decay=float(row["weight_decay"]),
        batch_size=int(row["batch_size"]),
        max_epochs=50,
        patience=5)

In [ ]:
def train_nuisance_model(nuisance, train_df, val_df, params, seed):
    
    # propensity score
    if nuisance == "e":
        # loaders
        train_loader, val_loader = make_nuisance_loaders(train_df, val_df, confounders, params["batch_size"])

        # init model
        model = ClassificationHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)

        # train model
        model, info = train_propensity(
            model,
            train_loader,
            val_loader,
            device,
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"],
            max_epochs=params["max_epochs"],
            patience=params["patience"],
            seed=seed)

        return model, info, "prop_model.pt"

    # response surfaces
    if nuisance in ["m0", "m1"]:

        # split training data
        treatment_value = 0 if nuisance == "m0" else 1
        train = train_df[train_df["T"] == treatment_value]
        val = val_df[val_df["T"] == treatment_value]

        # loaders
        train_loader, val_loader = make_nuisance_loaders(train, val, confounders, params["batch_size"])

        # init model
        model = RegressionHead(input_dim=input_dim, hidden_dim=params["hidden_dim"]).to(device)

        # train model
        model, info = train_response(
            model,
            train_loader,
            val_loader,
            device,
            lr=params["learning_rate"],
            weight_decay=params["weight_decay"],
            max_epochs=params["max_epochs"],
            patience=params["patience"],
            seed=seed)

        filename = f"mu{treatment_value}_model.pt"
        return model, info, filename

    raise ValueError(f"Unknown nuisance: {nuisance}")

#### train and store

In [ ]:
# load configs
configs = pd.read_csv(f'./configs/ihdp_nuisances.csv', index_col=0)

In [ ]:
# set checkpoint dir
out_dir = f'./chkpts/nuisances/'
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# loop over nuisances
for nuisance in ["e", "m0", "m1"]:

    # get hyperparameter settings
    params = get_nuisance_params(configs, nuisance)

    # loop over seeds
    for seed in range(50):

        # track progress
        print(f" -> Nuisance {nuisance}, seed {seed}")
        set_seed(seed)

        # set output dir
        ckpt_dir = Path(out_dir) / f"seed_{seed}"
        ckpt_dir.mkdir(parents=True, exist_ok=True)

        # get training data - no two stage splitting
        train_full = pd.read_csv(f'./data/datasets/replication_{seed}/train.csv')
        train_df, val_df = train_test_split(train_full,test_size=0.2,random_state=seed,shuffle=True)

        # train nuisance model
        set_seed(seed)
        model, info, filename = train_nuisance_model(nuisance=nuisance,train_df=train_df,val_df=val_df,params=params,seed=seed)

        # checkpoint
        torch.save(model.state_dict(), ckpt_dir / filename)